In [3]:
#以聊天智能体为例，没有记忆性
from langchain.agents import create_agent



agent=create_agent(
    model="deepseek-chat"
)



In [4]:
#建立一个最基本的聊天智能体，把我们的信息分享给模型
from langchain.messages import HumanMessage
response=agent.invoke(
    {"messages":[HumanMessage("你好，我叫胖猫，我最喜欢梦奇。")]}
)
print(response)

{'messages': [HumanMessage(content='你好，我叫胖猫，我最喜欢梦奇。', additional_kwargs={}, response_metadata={}, id='30fe0ce8-5b6b-4753-b224-4c72f4de2971'), AIMessage(content='你好胖猫！很高兴认识你！梦奇在《王者荣耀》里确实是个非常可爱的英雄，圆滚滚的造型和护盾机制让它又萌又能扛。喜欢梦奇的玩家通常性格温暖又带点调皮呢～  \n\n如果你有兴趣，我可以和你聊聊梦奇的出装思路、打法技巧，或者它背后的设计故事！你平时喜欢用梦奇打哪个位置？或者有什么关于它的趣事想分享吗？ 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 15, 'total_tokens': 108, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 15}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '3cb3c465-b94e-40f0-bb25-b55720193411', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dade0-db6c-7993-941d-d2517a23e985-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 93, '

In [6]:
#本次对话是知道我们提供的信息，为探究是否能够在下一次对话中记忆以上信息
response=agent.invoke(
    {"messages":[HumanMessage("我是谁，我最喜欢什么？")]}
)
for message in response["messages"]:
    message.pretty_print()#使用pretty_print不用再套一层print

================================ Human Message =================================

我是谁，我最喜欢什么？
================================== Ai Message ==================================

你好！我无法直接知道你的身份或喜好，因为每次对话对我来说都是全新的开始。不过，如果你愿意分享一些关于自己的信息（比如兴趣爱好、职业、喜欢的书籍/音乐等），我可以根据这些内容为你提供更个性化的交流或建议哦！ 😊

例如，你可以告诉我：
- 你最近热衷的事情
- 你喜欢的电影/游戏类型
- 你感兴趣的话题领域

期待通过对话更了解你！


In [7]:
#如果有记忆，应该是知道我是胖猫，喜欢梦奇
#但是“你好！我无法直接知道你的身份或喜好，因为每次对话对我来说都是全新的开始。不过，如果你愿意分享一些关于自己的信息（比如兴趣爱好、职业、喜欢的书籍/音乐等），我可以根据这些内容为你提供更个性化的交流或建议哦！ 😊”
"""
但是第一次调用时我们已经告知信息，但是他还是不知道说明现在的智能体是没有记忆的
本质原因是模型是无状态的，他每一次的调用都是独立的，不会保留之前的信息，想要有记忆最简单的方案是将历史对话的消息拼接在一起发送给模型
我们这里只发送了本次的请求/消息，历史的请求/消息没有发，所以没有记忆；只有本次请求与响应，没有历史的请求与响应
"""
"""
不同于会话机器人的记忆（即会话的历史消息），而智能体的记忆分为短期记忆short-term memory（当前任务/会话的上下文）、长期记忆long-term memory（跨任务/会话的经验与知识）
短期记忆不能简单认为是临时记忆，长期记忆也不能简单定义为永久的记忆，对于agent来讲区分记忆的不是记忆的存储方式（临时/永久记忆），而是记忆的作用域（当前任务或会话/跨任务或会话）
ep.对话机器人中历史消息/对话历史属于短期记忆，只记录了当前会话中用户与模型每一次对话的消息，只在当前对话生效
   智能体如果是带有工具的，在执行过程中，调用工具的信息、返回的结果、任务状态、任务执行时的临时数据都是在当前任务或者会话有效，都是属于短期记忆
   长期记忆一般是高级智能体的能力，可以是提前准备好的专业的数据，可作为模型外部的知识库（RAG那种），也可以是长期执行过程中不断积累的信息（用户的偏好、失败的经验），这些是与会话无关的，可以在任意会话都生效的记忆（即长期记忆）
"""
"""
短期记忆基于langchain的agentstate（智能体状态）来实现的，而会话历史（也就是消息列表）是agentstate的一部分（AgentState会记录我们与模型每一次交互产生的历史消息/任务执行状态/任务临时数据，默认只记录历史消息，其他需要自定义实现AgentState）
langchain提供了checkpoint对象来保存AgentState，每一次用户与AI交互都会生成一个快照，记录为一个checkpointer。两个作用，
1、实现会话记忆，每当用户发新休息，checkpointer都可以拿到最近的一次快照，拿到历史消息列表，把他和用户最新消息拼接得到一个完整的消息列表，把这个消息列表发给模型，模型基于历史消息回答，相当于拥有了会话记忆
2、实现时间穿梭，可以回退（到之前的某一次快照）
为方便管理会话，checkpointer中同一个对话的多个checkpointer星辰一个组，用同一个thread_id来标记；thread_id是用来区分会话，不同的会话有不同的thread_id，会话id
To add short-term memory (thread-level persistence) to an agent, you need to specify a checkpointer when creating an agent.为智能体添加短期记忆，我们需要在创建是指定一个checkpointer（短期记忆本质是基于agentstate实现的，而agentstate依赖于checkpointer来创建快照）
"""
"""
添加记忆的步骤：
1、导入初始化checkpointer
2、创建agent，指定checkpointer
3、调用checkpointer，指定thread_id
"""
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  #基于内存的实现/快照保存在内存中
#注意是langgraph包，不是langchain包;基于langchain开发智能体框架包含deepagence\langchain\langgraph,封装程度由高到低（复杂（多智能体、快捷（单智能体为主）、自己编排智能体内部工作流（但更灵活）
agent = create_agent(
    model="deepseek-chat",
    checkpointer=InMemorySaver(),#多一个参数checkpointer，值是对象初始化并赋值给checkpointer参数
)

agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Bob."}]},
    {"configurable": {"thread_id": "1"}},#多传一个字典参数，字典的键是"configurable"配置项（里面还是一个字典（“thread_id”线程id或者是会话id；同一个会话与ai的多次交互形成的历史消息列表会形成多个快照，作为一组去保存；不同会话作为不同的组里去保存；为了保证记忆，创建时指明绘画的id，这样才能知道是那一次对话，根据thread_id查到历史快照，找到历史消息，与最新消息做拼接发给模型；thread_id一定要去制定
)

{'messages': [HumanMessage(content='Hi! My name is Bob.', additional_kwargs={}, response_metadata={}, id='573793ca-5388-453d-9bc8-053fd41c6908'),
  AIMessage(content="Nice to meet you, Bob! I'm DeepSeek, an AI assistant created by DeepSeek. I'm here to help you with questions, tasks, or just have a friendly chat. \n\nWhat can I help you with today? Whether it's answering questions, helping with a project, brainstorming ideas, or anything else - I'm all ears! 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 11, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 11}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '78aa7fd9-6891-43f4-a8b7-9a735ebb0c23', 'finish_reason': 'stop', 'logprobs': Non

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  #基于内存的实现/快照保存在内存中
#注意是langgraph包，不是langchain包;基于langchain开发智能体框架包含deepagence\langchain\langgraph,封装程度由高到低（复杂（多智能体、快捷（单智能体为主）、自己编排智能体内部工作流（但更灵活）
agent = create_agent(
    model="deepseek-chat",
    checkpointer=InMemorySaver(),#多一个参数checkpointer，值是对象初始化并赋值给checkpointer参数
)

In [13]:
"""
为防止有多个对话，把配置项初始化为一个变量
"""
#设定thread_id作为会话标识
from langchain.messages import HumanMessage
config={"configurable":{"thread_id":"thread_1"}}
#第一次调用，告诉ai我的信息
response=agent.invoke(
    {"messages":[HumanMessage("你好，我叫蔡徐坤，我最喜欢打篮球。")]}
    ,config
)
print(response)

{'messages': [HumanMessage(content='你好，我叫蔡徐坤，我最喜欢打篮球。', additional_kwargs={}, response_metadata={}, id='fc084e6c-e7e9-4eb9-bfe8-e6386150d274'), AIMessage(content='你好，蔡徐坤！很高兴认识你！篮球确实是一项充满活力与团队精神的运动，无论是锻炼身体还是培养协作能力都非常棒。希望你能一直保持这份热爱，在球场上尽情享受运动的快乐～ 😊 如果以后想聊聊篮球技巧、赛事，或者任何你想分享的话题，我随时在这里哦！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 16, 'total_tokens': 84, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'ea5cbda8-d837-41d3-8457-324636c5e33d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dae22-a2ad-7fb2-9360-1d51b9052408-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 68, 'total_tokens': 84, 'input_tok

In [15]:
"""
第二次调用,询问我的信息，这次带上thread_id，唤起记忆
"""
response=agent.invoke(
    {"messages":[HumanMessage("我是谁，我最喜欢什么？")]},
    config#多传一个config，现在thread_id是一致的
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

你好，我叫蔡徐坤，我最喜欢打篮球。
================================== Ai Message ==================================

你好，蔡徐坤！很高兴认识你！篮球确实是一项充满活力与团队精神的运动，无论是锻炼身体还是培养协作能力都非常棒。希望你能一直保持这份热爱，在球场上尽情享受运动的快乐～ 😊 如果以后想聊聊篮球技巧、赛事，或者任何你想分享的话题，我随时在这里哦！
================================ Human Message =================================

我是谁，我最喜欢什么？
================================== Ai Message ==================================

你是蔡徐坤，你最喜欢打篮球！ 😊 记得继续享受在球场上挥洒汗水的快乐呀～ 如果需要聊聊篮球相关的话题，我随时都在哦！
================================ Human Message =================================

我是谁，我最喜欢什么？
================================== Ai Message ==================================

你是蔡徐坤，你最喜欢打篮球！ 🌟 保持这份热爱，继续在球场上发光吧～ 如果需要讨论篮球技巧、赛事，或者任何想聊的内容，我都很乐意陪你交流！


In [16]:
print(response)
#包含第一次对话的内容/第一次ai的回答和本次对话的内容（问的问题）/ai的回答

{'messages': [HumanMessage(content='你好，我叫蔡徐坤，我最喜欢打篮球。', additional_kwargs={}, response_metadata={}, id='fc084e6c-e7e9-4eb9-bfe8-e6386150d274'), AIMessage(content='你好，蔡徐坤！很高兴认识你！篮球确实是一项充满活力与团队精神的运动，无论是锻炼身体还是培养协作能力都非常棒。希望你能一直保持这份热爱，在球场上尽情享受运动的快乐～ 😊 如果以后想聊聊篮球技巧、赛事，或者任何你想分享的话题，我随时在这里哦！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 16, 'total_tokens': 84, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'ea5cbda8-d837-41d3-8457-324636c5e33d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dae22-a2ad-7fb2-9360-1d51b9052408-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 68, 'total_tokens': 84, 'input_tok

In [ ]:
"""
每一次我们与ai交互时，会把用户的消息和ai返回的消息保存起来形成快照
但我们再次调用时。他会调用快照，拿到历史消息，把历史消息与用户最新的提问进行拼接，发给模型，这样模型根据历史（消息列表）回答，体现记忆性
"""

In [18]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
config1={"configurable":{"thread_id":"thread_1"}}
config2={"configurable":{"thread_id":"thread_2"}}
agent=create_agent(
    model="deepseek-chat",
    checkpointer=InMemorySaver(),
)
response=agent.invoke(
    {"messages":[HumanMessage("我是南京邮电大学信息安全专业的学生")]},
    config1
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我是南京邮电大学信息安全专业的学生
================================== Ai Message ==================================

你好！南京邮电大学的信息安全专业非常出色，尤其是在通信与网络安全的交叉领域，南邮有着很强的学科优势和行业声誉。作为这个专业的学生，你正处在一个充满机遇和挑战的领域。

以下是一些可能对你有帮助的方向和建议：

### 1. **专业核心优势**
   - **通信安全特色**：南邮的背景意味着你在**网络通信协议安全、移动安全、物联网安全**等方面可能接触到更多资源。可以重点关注如5G安全、无线网络安全等方向。
   - **课程体系**：确保扎实掌握**密码学、网络攻防、系统安全、渗透测试、安全编程**等核心课程。南邮的实践环节（如实验、课程设计）通常比较丰富，多动手操作。

### 2. **技能拓展建议**
   - **技术能力**：
     - 编程：至少精通一门语言（如Python、C/C++），并了解脚本语言（Bash/PowerShell）。
     - 工具：熟悉Wireshark、Metasploit、Burp Suite、IDA Pro等安全工具。
     - 认证：可以考虑考取**CISP、CISSP、Security+** 等认证，或技术类证书如**OSCP（渗透测试）**。
   - **实践机会**：
     - 参与CTF比赛：南邮有活跃的CTF战队（如X1cT34m），参赛能快速提升实战能力。
     - 漏洞研究：关注SRC（安全应急响应中心）平台，尝试提交漏洞。
     - 实验室项目：加入学校的网络安全实验室（如江苏省无线通信重点实验室），参与科研或项目。

### 3. **未来方向参考**
   - **就业热门领域**：
     - 网络安全企业：如华为、奇安信、绿盟、天融信等。
     - 互联网公司：腾讯、阿里、字节跳动的安全部门（安全工程师、安全研发）。
     - 通信运营商/机构：中国移动、电信、联通的安全岗位，或国家互联网应急中心（C

In [19]:
response=agent.invoke(
    {"messages":[HumanMessage("请问我的专业是什么？")]},
    config1
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我是南京邮电大学信息安全专业的学生
================================== Ai Message ==================================

你好！南京邮电大学的信息安全专业非常出色，尤其是在通信与网络安全的交叉领域，南邮有着很强的学科优势和行业声誉。作为这个专业的学生，你正处在一个充满机遇和挑战的领域。

以下是一些可能对你有帮助的方向和建议：

### 1. **专业核心优势**
   - **通信安全特色**：南邮的背景意味着你在**网络通信协议安全、移动安全、物联网安全**等方面可能接触到更多资源。可以重点关注如5G安全、无线网络安全等方向。
   - **课程体系**：确保扎实掌握**密码学、网络攻防、系统安全、渗透测试、安全编程**等核心课程。南邮的实践环节（如实验、课程设计）通常比较丰富，多动手操作。

### 2. **技能拓展建议**
   - **技术能力**：
     - 编程：至少精通一门语言（如Python、C/C++），并了解脚本语言（Bash/PowerShell）。
     - 工具：熟悉Wireshark、Metasploit、Burp Suite、IDA Pro等安全工具。
     - 认证：可以考虑考取**CISP、CISSP、Security+** 等认证，或技术类证书如**OSCP（渗透测试）**。
   - **实践机会**：
     - 参与CTF比赛：南邮有活跃的CTF战队（如X1cT34m），参赛能快速提升实战能力。
     - 漏洞研究：关注SRC（安全应急响应中心）平台，尝试提交漏洞。
     - 实验室项目：加入学校的网络安全实验室（如江苏省无线通信重点实验室），参与科研或项目。

### 3. **未来方向参考**
   - **就业热门领域**：
     - 网络安全企业：如华为、奇安信、绿盟、天融信等。
     - 互联网公司：腾讯、阿里、字节跳动的安全部门（安全工程师、安全研发）。
     - 通信运营商/机构：中国移动、电信、联通的安全岗位，或国家互联网应急中心（C

In [20]:
response=agent.invoke(
    {"messages":[HumanMessage("请问我的专业是什么？")]},
    config2
)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

请问我的专业是什么？
================================== Ai Message ==================================

我无法直接访问您的个人信息，因此不知道您的专业背景。不过，如果您愿意告诉我，我可以根据您提供的专业领域，为您解答相关问题、推荐学习资源或探讨行业动态。请随时分享您的专业或兴趣方向！ 😊
